<a href="https://colab.research.google.com/github/chamudithamk/ME422-B2-Lab-Group/blob/main/Control/E_20_190_Rigid_body_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Derivation: Twin Rotor System Model

## Setup and Assumptions

Consider a rigid body (the twin rotor platform) constrained so that it *cannot rotate about its $\mathbf{b}_2$ body axis*. The body has:
- Inertia tensor $\mathbb{I}$ in the body frame
- Two rotors producing thrust forces at angles $\alpha$ and $\beta$ from $\mathbf{b}_1$, mounted in the $\mathbf{b}_1$–$\mathbf{b}_3$ plane
- Angular momentum $\pi = \mathbb{I}^R \omega$ in the spatial frame, where $\mathbb{I}^R = R\mathbb{I}R^T$

---

## Step 1: Kinematics on SO(3)

The orientation $R \in SO(3)$ satisfies the standard kinematic equation. For a rigid body with spatial angular velocity $\omega$, the time derivative of the rotation matrix is:

$$\boxed{\dot{R} = \widehat{\omega}R}$$

where $\widehat{\omega}$ is the $3\times 3$ skew-symmetric matrix such that $\widehat{\omega}v = \omega \times v$ for any vector $v$. This is the standard result from differentiating $RR^T = I$.

---

## Step 2: Angular Momentum Dynamics

Newton–Euler in the spatial frame gives:

$$\dot{\pi} = \tau_{\text{total}}$$

Splitting all applied torques into the constraint part $\tau^e$ (reaction moment enforcing the $\mathbf{b}_2$ constraint) and the control part $\tau^u$ (from the rotors):

$$\boxed{\dot{\pi} = \tau^e + \tau^u}$$

The spatial angular velocity is recovered from $\pi = \mathbb{I}^R\omega$, hence $\omega = (\mathbb{I}^R)^{-1}\pi$.

---

## Step 3: Deriving the Control Torque $\tau^u$

Each rotor $i$ produces a scalar thrust $u_i$ directed at a fixed angle in the $\mathbf{b}_1$–$\mathbf{b}_3$ plane of the body frame. In the body frame, the thrust directions are:

$$f_1^{\text{body}} = \begin{bmatrix}\cos\alpha \\ 0 \\ \sin\alpha\end{bmatrix}, \qquad f_2^{\text{body}} = \begin{bmatrix}-\cos\beta \\ 0 \\ -\sin\beta\end{bmatrix}$$

The signs reflect the counter-acting geometry of a twin-rotor (the two rotors apply moments in opposing senses). These forces produce torques acting on the body. Collecting both:

$$\tau^u_{\text{body}} = \begin{bmatrix}\cos\alpha & -\cos\beta \\ 0 & 0 \\ \sin\alpha & -\sin\beta\end{bmatrix}\begin{bmatrix}u_1 \\ u_2\end{bmatrix} = \underbrace{\begin{bmatrix}1&0\\0&0\\0&1\end{bmatrix}}{3\times 2}\underbrace{\begin{bmatrix}\cos\alpha & -\cos\beta \\ \sin\alpha & -\sin\beta\end{bmatrix}}{2\times 2}\begin{bmatrix}u_1\\u_2\end{bmatrix}$$

The selection matrix explicitly zeros out any direct actuation along $\mathbf{b}_2$, consistent with the rotor geometry. Rotating to the spatial frame:

$$\boxed{\tau^u = R\begin{bmatrix}1&0\\0&0\\0&1\end{bmatrix}\begin{bmatrix}\cos\alpha & -\cos\beta\\\sin\alpha&-\sin\beta\end{bmatrix}\begin{bmatrix}u_1\\u_2\end{bmatrix}}$$

---

## Step 4: Deriving the Constraint Torque $\tau^e$

The mechanical constraint prevents rotation about $\mathbf{b}_2$, i.e.:

$$e_2^T \Omega = 0 \quad \text{and} \quad e_2^T \dot{\Omega} = 0 \quad \text{for all time}$$

where $\Omega = R^T\omega$ is the body-frame angular velocity. The body-frame Euler equation is obtained by transforming $\dot{\pi} = \tau^e + \tau^u$ into the body frame. Using $\pi = R\,\mathbb{I}\,\Omega$ and differentiating:

$$\dot{\pi} = \dot{R}\,\mathbb{I}\,\Omega + R\,\mathbb{I}\,\dot{\Omega} = \widehat{\omega}R\,\mathbb{I}\,\Omega + R\,\mathbb{I}\,\dot{\Omega}$$

Pre-multiplying by $R^T$ and using $R^T\widehat{\omega}R = \widehat{\Omega}$ (the adjoint identity):

$$\mathbb{I}\dot{\Omega} + \widehat{\Omega}\,\mathbb{I}\,\Omega = R^T(\tau^e + \tau^u)$$

Recognising $\widehat{\Omega}\,\mathbb{I}\,\Omega = \Omega \times \mathbb{I}\Omega$, the full body-frame equation is:

$$\mathbb{I}\dot{\Omega} + \Omega\times\mathbb{I}\Omega = R^T\tau^e + R^T\tau^u$$

Since the rotors produce *no torque along $\mathbf{b}_2$* (the selection matrix zeros that row), we have $e_2^T R^T \tau^u = 0$. Projecting the equation onto $\mathbf{b}_2$ via $e_2^T$:

$$e_2^T\!\left(\mathbb{I}\dot{\Omega} + \Omega\times\mathbb{I}\Omega\right) = e_2^T R^T \tau^e$$

The constraint torque acts purely along $\mathbf{b}_2$ in the body frame, so $R^T\tau^e = \begin{bmatrix}0 & T_2 & 0\end{bmatrix}^T$, giving $e_2^T R^T\tau^e = T_2$. Therefore:

$$\boxed{T_2 = e_2^T\!\left(\Omega\times\mathbb{I}\Omega + \mathbb{I}\dot{\Omega}\right)}$$

and rotating back to the spatial frame:

$$\boxed{\tau^e = R\begin{bmatrix}0\\T_2\\0\end{bmatrix}}$$

# 2. Simulation


In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import solve_ivp

In [2]:


# ====================== Physics & System Parameters =========================
I1, I2, I3 = 1.0, 2.0, 1.5  # Inertia tensor components


# ============================= Inputs =======================================
def get_current_inputs(t):
    # --- Uncomment ONLY ONE for testing ---

    # return [0.5 * np.sin(2 * t), -0.2 * np.cos(t)]  # Frequency Response
    # return [1.0, 1.0]                               # Step Response
    # return [1.0, 0.0]                               # Pitch Test
    return [0.0, 1.0]                                 # Yaw Test


def u1_torque(t):
    u1, _ = get_current_inputs(t)
    return u1


def u2_torque(t):
    _, u2 = get_current_inputs(t)
    return u2


# ========================= Differential Equations ===========================
def twin_rotor_dynamics(t, state):
    theta, phi, dtheta, dphi = state

    u1, u2 = get_current_inputs(t)

    cos_phi = np.cos(phi)
    if abs(cos_phi) < 1e-5:
        cos_phi = 1e-5 if cos_phi == 0 else 1e-5 * np.sign(cos_phi)

    ddphi = (u1 - (I3 - I2) * (dtheta ** 2) * np.sin(phi) * cos_phi) / I1
    ddtheta = (-u2 - (I2 - I1 - I3) * dtheta * dphi * np.sin(phi)) / (I3 * cos_phi)

    return [dtheta, dphi, ddtheta, ddphi]


# ============================ Simulation Run ================================
t_span = (0, 10)
t_eval = np.linspace(*t_span, 300)

y0 = [0.1, 0.0, 0.0, 0.0]  # Initial conditions

solution = solve_ivp(twin_rotor_dynamics, t_span, y0, t_eval=t_eval, method='RK45')

theta_vals, phi_vals = solution.y[0], solution.y[1]


# =================== Kinematics & 3D Geometry ===============================
def get_rotation_matrix(theta, phi):
    cth, sth = np.cos(theta), np.sin(theta)
    cph, sph = np.cos(phi), np.sin(phi)

    R3 = np.array([
        [cth, -sth, 0],
        [sth,  cth, 0],
        [0,    0,   1]
    ])

    R1 = np.array([
        [1, 0,   0],
        [0, cph, -sph],
        [0, sph,  cph]
    ])

    return R3 @ R1


frames = []
L_support, L_beam = 2.0, 3.0

for th, ph in zip(theta_vals, phi_vals):
    R = get_rotation_matrix(th, ph)

    support_line = np.array([[0, 0, 0], [0, 0, -L_support]])

    beam_local = np.array([[0, -L_beam / 2, 0], [0, L_beam / 2, 0]])
    beam_global = (R @ beam_local.T).T

    fan1_local = np.array([[0.5, -L_beam / 2, 0], [-0.5, -L_beam / 2, 0]])
    fan2_local = np.array([[0, L_beam / 2, 0.5], [0, L_beam / 2, -0.5]])

    fan1_global = (R @ fan1_local.T).T
    fan2_global = (R @ fan2_local.T).T

    x = [
        support_line[0, 0], support_line[1, 0], None,
        beam_global[0, 0], beam_global[1, 0], None,
        fan1_global[0, 0], fan1_global[1, 0], None,
        fan2_global[0, 0], fan2_global[1, 0]
    ]

    y = [
        support_line[0, 1], support_line[1, 1], None,
        beam_global[0, 1], beam_global[1, 1], None,
        fan1_global[0, 1], fan1_global[1, 1], None,
        fan2_global[0, 1], fan2_global[1, 1]
    ]

    z = [
        support_line[0, 2], support_line[1, 2], None,
        beam_global[0, 2], beam_global[1, 2], None,
        fan1_global[0, 2], fan1_global[1, 2], None,
        fan2_global[0, 2], fan2_global[1, 2]
    ]

    frames.append(go.Frame(data=[go.Scatter3d(x=x, y=y, z=z)]))


# =========================== Animation Setup ================================
fig = go.Figure(
    data=[go.Scatter3d(
        x=frames[0].data[0].x,
        y=frames[0].data[0].y,
        z=frames[0].data[0].z,
        mode='lines',
        line=dict(color='blue', width=6)
    )],
    layout=go.Layout(
        title="Twin Rotor System Dynamics",
        scene=dict(
            xaxis=dict(range=[-3, 3], title='X (Earth)'),
            yaxis=dict(range=[-3, 3], title='Y (Earth)'),
            zaxis=dict(range=[-3, 3], title='Z (Earth)'),
            aspectmode='cube'
        ),
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(
                label="Play",
                method="animate",
                args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]
            )]
        )]
    ),
    frames=frames
)

fig.show()